# Início

In [ ]:
pip install torchmetrics

In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchmetrics.functional.classification import multiclass_f1_score
from sklearn.metrics import f1_score

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import copy
from copy import deepcopy
from tqdm import tqdm
import time
import os
from scipy.signal import butter, sosfiltfilt, resample_poly
from scipy.interpolate import PchipInterpolator
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [4]:
def recortar_janelas(acc, J, passo):
    N = acc.shape[0]
    Nj = (N - J) // passo + 1
    janelas = np.zeros((Nj, J, 3))
    for i in range(Nj):
        janelas[i] = acc[i * passo:i * passo + J]
    return janelas

In [5]:
def scipy_interpolate_signal(series, method='pchip'):
    y = series.values
    x = np.arange(len(y))

    # Identifica índices válidos (não-NaN)
    valid_mask = ~np.isnan(y)

    # Se todos forem NaN ou houver apenas 1 ponto válido, retorna como está
    if valid_mask.sum() < 2:
        return series

    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    interpolator = PchipInterpolator(x_valid, y_valid, extrapolate=True)
    y_filled = interpolator(x)

    return pd.Series(y_filled, index=series.index)

In [6]:
actisP = ['lying', 'sitting', 'standing', 'walking', 'running', 'cycling',
          'Nordic', 'watching TV', 'computer', 'car driving', 'ascending stairs', 'descending stairs',
          'vacuum', 'ironing', 'folding', 'house cleaning', 'soccer', 'rope jumping']
activity_id = [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24]

In [7]:
# 1. Definir os sufixos das 17 variáveis de cada sensor IMU
imu_features = [
    'temp',
    'acc16_x', 'acc16_y', 'acc16_z',  # Aceleração (escala ±16g)
    'acc6_x', 'acc6_y', 'acc6_z',     # Aceleração (escala ±6g)
    'gyro_x', 'gyro_y', 'gyro_z',     # Giroscópio
    'mag_x', 'mag_y', 'mag_z',        # Magnetômetro
    'orient_x', 'orient_y', 'orient_z', 'orient_w' # Orientação (Quatérnios)
]
columns = ['timestamp', 'activity_id', 'heart_rate']
users = ['10'+ x for x in np.arange(1,10).astype(str)]
posis = ['hand', 'chest', 'ankle']
for sensor_location in posis:
    for feature in imu_features:
        columns.append(f'IMU_{sensor_location}_{feature}')
acc_cols = [
    'IMU_ankle_acc16_x', 'IMU_ankle_acc16_y', 'IMU_ankle_acc16_z',
    'IMU_hand_acc16_x', 'IMU_hand_acc16_y', 'IMU_hand_acc16_z',
    'IMU_chest_acc16_x', 'IMU_chest_acc16_y', 'IMU_chest_acc16_z']

file_path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/PAMAP2_Dataset/Protocol/subject103.dat'
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)
df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()
for col in acc_cols:
    df[col] = df.groupby('activity_block')[col].transform(
        lambda group: scipy_interpolate_signal(group, method='pchip')
    )
df = df.drop(columns=['activity_block'])

In [8]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/PAMAP2_Dataset/'
data = np.load(path+'Xydata.npz')
Xdata = data['Xdata']/20 # /160
ydata = data['ydata']
valid = list(np.unique(ydata[:,1]))
for i in range(ydata.shape[0]):
    ydata[i,1] = valid.index(ydata[i,1])

In [9]:
def spectral_perturb(x, gain_db=6., n_knots=9, generator=None, preserve_dc=True, eps=1e-12):
    """x: torch.Tensor (B,3,J), retorna mesma forma, dtype e energia por janela.
    Resposta em dB interpolada linearmente em frequencia, compartilhada nos 3 eixos.
    Os limites +/-gain_db valem ANTES da renormalizacao de energia.
    """
    if gain_db < 0: raise ValueError('gain_db deve ser >= 0')
    if x.ndim != 3: raise ValueError('Esperado (B,3,J)')
    if not x.is_floating_point(): raise TypeError('Entrada deve ser float')
    B,C,J=x.shape
    if J < 4 or n_knots < 2: raise ValueError('J>=4 e n_knots>=2')
    if gain_db == 0: return x.clone()
    # Gerador CPU permite reproducibilidade independentemente do dispositivo.
    knots = (2*torch.rand((B,1,n_knots),generator=generator)-1)*gain_db
    if preserve_dc: knots[:,:,0]=0.
    # interpolacao de magnitudes em dB ao longo dos bins positivos da rFFT
    db = torch.nn.functional.interpolate(knots, size=J//2+1, mode='linear',align_corners=True)
    gain = torch.pow(torch.tensor(10.,dtype=x.dtype,device=x.device), db.to(x.device,dtype=x.dtype)/20)
    spectrum = torch.fft.rfft(x,dim=-1)
    y = torch.fft.irfft(spectrum*gain,n=J,dim=-1)
    e0 = x.square().sum(dim=(1,2),keepdim=True)
    e1 = y.square().sum(dim=(1,2),keepdim=True)
    scale = torch.sqrt(e0/e1.clamp_min(eps))
    return y*scale

# Preprocessamento PAMAP2

In [ ]:
users = ['10'+ x for x in np.arange(1,10).astype(str)]
posis = ['hand', 'chest', 'ankle']
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/PAMAP2_Dataset/Protocol/'
J = 150
Xdata = []
ydata = []

for user in users:
    file_path = path+'subject'+user+'.dat'
    df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)
    df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()
    for col in acc_cols:
        df[col] = df.groupby('activity_block')[col].transform(
            lambda group: scipy_interpolate_signal(group, method='pchip')
        )
    df = df.drop(columns=['activity_block'])
    for posi in posis:
        for act in activity_id:
            df_run = df[df['activity_id'] == act]
            if df_run.size == 0:
                continue
            x = df_run[f'IMU_{posi}_acc16_x'].values
            x = resample_poly(x, 1, 2)
            y = df_run[f'IMU_{posi}_acc16_y'].values
            y = resample_poly(y, 1, 2)
            z = df_run[f'IMU_{posi}_acc16_z'].values
            z = resample_poly(z, 1, 2)
            acc = np.stack((x,y,z), axis=1)
            janelas = recortar_janelas(acc, J, J)
            rotulos = np.full((janelas.shape[0], 3), [posis.index(posi), act, users.index(user)], dtype=int)
            Xdata.append(janelas)
            ydata.append(rotulos)
            print(janelas.shape[0], [posi, act, user])
Xdata = np.concatenate(Xdata, axis=0)
# Xdata = Xdata/160    # Normalização entre 1 e -1
ydata = np.concatenate(ydata, axis=0)
Xdata.shape, ydata.shape

90 ['hand', 1, '101']
78 ['hand', 2, '101']
72 ['hand', 3, '101']
74 ['hand', 4, '101']
70 ['hand', 5, '101']
78 ['hand', 6, '101']
67 ['hand', 7, '101']
52 ['hand', 12, '101']
49 ['hand', 13, '101']
76 ['hand', 16, '101']
78 ['hand', 17, '101']
43 ['hand', 24, '101']
90 ['chest', 1, '101']
78 ['chest', 2, '101']
72 ['chest', 3, '101']
74 ['chest', 4, '101']
70 ['chest', 5, '101']
78 ['chest', 6, '101']
67 ['chest', 7, '101']
52 ['chest', 12, '101']
49 ['chest', 13, '101']
76 ['chest', 16, '101']
78 ['chest', 17, '101']
43 ['chest', 24, '101']
90 ['ankle', 1, '101']
78 ['ankle', 2, '101']
72 ['ankle', 3, '101']
74 ['ankle', 4, '101']
70 ['ankle', 5, '101']
78 ['ankle', 6, '101']
67 ['ankle', 7, '101']
52 ['ankle', 12, '101']
49 ['ankle', 13, '101']
76 ['ankle', 16, '101']
78 ['ankle', 17, '101']
43 ['ankle', 24, '101']
78 ['hand', 1, '102']
74 ['hand', 2, '102']
85 ['hand', 3, '102']
108 ['hand', 4, '102']
30 ['hand', 5, '102']
83 ['hand', 6, '102']
99 ['hand', 7, '102']
57 ['hand', 12

((19278, 150, 3), (19278, 3))

In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/PAMAP2_Dataset/'
np.savez(path+'Xydata.npz', Xdata=Xdata, ydata=ydata)

In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/PAMAP2_Dataset/'
data = np.load(path+'Xydata.npz')
Xdata = data['Xdata']/160
ydata = data['ydata']

# Modelo Chang

In [11]:
class ChangEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3)
        self.inst1 = nn.InstanceNorm1d(16, affine=True)
        self.drop1 = nn.Dropout(p=0.2)

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3)
        self.inst2 = nn.InstanceNorm1d(16, affine=True)
        self.drop2 = nn.Dropout(p=0.2)

        self.conv3 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, stride=4)
        self.inst3 = nn.InstanceNorm1d(32, affine=True)
        self.drop3 = nn.Dropout(p=0.2)

        self.conv4 = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=3, stride=1)
        self.inst4 = nn.InstanceNorm1d(32, affine=True)
        self.drop4 = nn.Dropout(p=0.2)

        self.conv5 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=4)
        self.inst5 = nn.InstanceNorm1d(64, affine=True)
        self.drop5 = nn.Dropout(p=0.2)

        self.conv6 = nn.Conv1d(in_channels=64, out_channels=100, kernel_size=5, stride=1)

        self.relu = nn.LeakyReLU(0.3)
        self.glap = nn.AvgPool1d(kernel_size=4)

    def forward(self, x):
        # (N,T,C) -> (N,C,T)
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.inst1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.inst2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.inst3(x)
        x = self.drop3(x)

        x = self.conv4(x)
        x = self.relu(x)
        x = self.inst4(x)
        x = self.drop4(x)

        x = self.conv5(x)
        x = self.relu(x)
        x = self.inst5(x)
        x = self.drop5(x)

        x = self.conv6(x)
        x = self.relu(x)

        x = self.glap(x)
        x = x.flatten(start_dim=1)

        logits = x
        return logits

In [12]:
class ChangClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.densa = nn.Linear(in_features=100, out_features=12)

    def forward(self, x):
        logits = self.densa(x)
        return logits

# Funções de treinamento

In [13]:
@torch.no_grad()
def evaluate(encoder, classifier, loader, loss_fn, device):
    encoder.eval()
    classifier.eval()
    total_loss = 0
    total_samples = 0
    y_true = []
    y_pred = []

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = classifier(encoder(X))
        loss = loss_fn(logits, y)
        total_loss += loss.item() * len(y)
        total_samples += len(y)
        pred = torch.argmax(logits, dim=1)
        y_true.append(y)
        y_pred.append(pred)

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)
    f1 = multiclass_f1_score(y_pred, y_true, num_classes=12, average="macro").item()

    return total_loss / total_samples, f1

In [14]:
def train_model(train_loader, val_loader, device, n_epochs):
    encoder = ChangEncoder().to(device)
    classifier = ChangClassifier().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(classifier.parameters()), lr=1e-3)
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_f1": [],
        "val_f1": []
    }
    best_f1 = -1
    best_encoder = None
    best_classifier = None

    for epoch in range(n_epochs):
        encoder.train()
        classifier.train()
        running_loss = 0
        n_samples = 0
        bar = tqdm(train_loader)

        for X, y in bar:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = classifier(encoder(X))
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)
            n_samples += len(y)
            bar.set_description(f"Epoch {epoch+1}")
            bar.set_postfix(loss=loss.item())

        train_loss, train_f1 = evaluate(
            encoder,
            classifier,
            train_loader,
            loss_fn,
            device
        )

        val_loss, val_f1 = evaluate(
            encoder,
            classifier,
            val_loader,
            loss_fn,
            device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch {epoch+1:3d} | "
            f"Train F1={train_f1:.4f} | "
            f"Val F1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_encoder = deepcopy(encoder)
            best_classifier = deepcopy(classifier)

    return best_encoder, best_classifier, history

# Teste baseline

In [ ]:
i, posi = 1, 'chest'
inds = ydata[:,0]==i
X = Xdata[inds]
y = ydata[inds][:,1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
enc, cla, history = train_model(train_loader, test_loader, device, n_epochs=100)

In [ ]:
test_loss, test_f1 = evaluate(
    enc,
    cla,
    test_loader,
    nn.CrossEntropyLoss(),
    device
)

print(test_loss)
print(test_f1)

0.7113335002822045
0.8131365180015564


In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/baseline_10_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
for rep in range(10):
    print(rep)
    for i, posi in enumerate(posis):
        inds = ydata[:,0]==i
        X = Xdata[inds]
        y = ydata[inds][:,1]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
        X_train = torch.tensor(X_train, dtype=torch.float32)
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_train = torch.tensor(y_train, dtype=torch.long)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        enc, cla, history = train_model(train_loader, test_loader, device, n_epochs=100)
        nome = str(rep)+'_baseline_encoder_'+posi+'.pth'
        torch.save(enc.state_dict(), path + nome)
        nome = str(rep)+'_baseline_classifier_'+posi+'.pth'
        torch.save(cla.state_dict(), path + nome)

In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/baseline_10_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
resultados = np.zeros((10,3,3))
for rep in tqdm(range(10)):
    for i, posi in enumerate(posis):
        enc = ChangEncoder().to(device)
        cla = ChangClassifier().to(device)
        nome = str(rep)+'_baseline_encoder_'+posi+'.pth'
        enc.load_state_dict(torch.load(path + nome))
        nome = str(rep)+'_baseline_classifier_'+posi+'.pth'
        cla.load_state_dict(torch.load(path + nome))
        for j in range(3):
            inds = ydata[:,0]==j
            X = Xdata[inds]
            y = ydata[inds][:,1]
            _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
            X_test  = torch.tensor(X_test, dtype=torch.float32)
            y_test  = torch.tensor(y_test, dtype=torch.long)
            test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
            _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
            resultados[rep,i,j] = test_f1
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
print(), print()
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

100%|██████████| 10/10 [00:29<00:00,  2.92s/it]



hand:	76 +- 1 | 28 +- 2 | 15 +- 3 | 
chest:	21 +- 1 | 82 +- 1 | 11 +- 2 | 
ankle:	15 +- 2 | 14 +- 3 | 80 +- 1 | 


# Exp Filtros aleatórios

In [ ]:
gain_db=24.
n_knots=20
generator=None#torch.Generator().manual_seed(43)
preserve_dc=True
eps=1e-12
inds = ydata[:,1]==3
x = torch.tensor(Xdata[inds,:,:]).transpose(1, 2)
if not x.is_floating_point(): raise TypeError('Entrada deve ser float')
B,C,J=x.shape
# Gerador CPU permite reproducibilidade independentemente do dispositivo.
knots = (2*torch.rand((B,1,n_knots),generator=generator)-1)*gain_db
if preserve_dc: knots[:,:,0]=0.
# interpolacao de magnitudes em dB ao longo dos bins positivos da rFFT
db = torch.nn.functional.interpolate(knots, size=J//2+1, mode='linear',align_corners=True)
gain = torch.pow(torch.tensor(10.,dtype=x.dtype,device=x.device), db.to(x.device,dtype=x.dtype)/20)
spectrum = torch.fft.rfft(x,dim=-1)
y = torch.fft.irfft(spectrum*gain,n=J,dim=-1)
e0 = x.square().sum(dim=(1,2),keepdim=True)
e1 = y.square().sum(dim=(1,2),keepdim=True)
scale = torch.sqrt(e0/e1.clamp_min(eps))
saida = y*scale
fig = go.Figure()
fig.add_trace(go.Scatter(y=x[10,0,:].numpy().reshape(-1), mode='lines', name='x0'))
fig.add_trace(go.Scatter(y=x[10,1,:].numpy().reshape(-1), mode='lines', name='y0'))
fig.add_trace(go.Scatter(y=x[10,2,:].numpy().reshape(-1), mode='lines', name='z0'))
fig.add_trace(go.Scatter(y=y[10,0,:].numpy().reshape(-1), mode='lines', name='x1'))
fig.add_trace(go.Scatter(y=y[10,1,:].numpy().reshape(-1), mode='lines', name='y1'))
fig.add_trace(go.Scatter(y=y[10,2,:].numpy().reshape(-1), mode='lines', name='z1'))
fig.show()

In [14]:
def spectral_perturb(x, gain_db=6., n_knots=9, generator=None, preserve_dc=True, eps=1e-12):
    if gain_db < 0:
        raise ValueError('gain_db deve ser >= 0')
    if x.ndim != 3:
        raise ValueError('Esperado (B,3,J)')
    if not np.issubdtype(x.dtype, np.floating):
        raise TypeError('Entrada deve ser float')
    B, C, J = x.shape
    if J < 4 or n_knots < 2:
        raise ValueError('J>=4 e n_knots>=2')
    if gain_db == 0:
        return x.copy()
    if generator is None:
        generator = np.random.default_rng()
    # Ganhos aleatórios nos nós de interpolação
    knots = (2 * generator.random((B, n_knots)) - 1) * gain_db
    if preserve_dc:
        knots[:, 0] = 0.
    # Interpolação linear nos bins positivos da rFFT
    n_freq = J // 2 + 1
    pos = np.linspace(0, n_knots - 1, n_freq)
    i0 = np.floor(pos).astype(int)
    i1 = np.minimum(i0 + 1, n_knots - 1)
    w = pos - i0
    db = (1 - w) * knots[:, i0] + w * knots[:, i1]
    # Ganho linear, compartilhado nos três eixos
    gain = (10.0 ** (db / 20.0))[:, None, :]
    # Perturbação espectral
    spectrum = np.fft.rfft(x, axis=-1)
    y = np.fft.irfft(spectrum * gain, n=J, axis=-1)
    # Preservação da energia de cada janela
    e0 = np.sum(x**2, axis=(1, 2), keepdims=True)
    e1 = np.sum(y**2, axis=(1, 2), keepdims=True)
    scale = np.sqrt(e0 / np.maximum(e1, eps))
    return (y * scale).astype(x.dtype, copy=False)

In [91]:
inds = (ydata[:,1]==3) & (ydata[:,0]==0) & (ydata[:,2]==0)
X = Xdata[inds]
y = ydata[inds]
Xp = Xpert[inds]
print(y[i], X.shape)
fig = go.Figure()
fig.add_trace(go.Scatter(y=X[i,:,0], mode='lines', name='x0'))
fig.add_trace(go.Scatter(y=X[i,:,1], mode='lines', name='y0'))
fig.add_trace(go.Scatter(y=X[i,:,2], mode='lines', name='z0'))
fig.add_trace(go.Scatter(y=Xp[i,:,0], mode='lines', name='x1'))
fig.add_trace(go.Scatter(y=Xp[i,:,1], mode='lines', name='y1'))
fig.add_trace(go.Scatter(y=Xp[i,:,2], mode='lines', name='z1'))
fig.show()

[0 3 0] (74, 150, 3)


In [89]:
inds = (ydata[:,1]==3) & (ydata[:,0]==1) & (ydata[:,2]==0)
X = Xdata[inds]
y = ydata[inds]
Xp = Xpert[inds]
print(y[i], X.shape)
fig = go.Figure()
fig.add_trace(go.Scatter(y=X[i,:,0], mode='lines', name='x0'))
fig.add_trace(go.Scatter(y=X[i,:,1], mode='lines', name='y0'))
fig.add_trace(go.Scatter(y=X[i,:,2], mode='lines', name='z0'))
fig.add_trace(go.Scatter(y=Xp[i,:,0], mode='lines', name='x1'))
fig.add_trace(go.Scatter(y=Xp[i,:,1], mode='lines', name='y1'))
fig.add_trace(go.Scatter(y=Xp[i,:,2], mode='lines', name='z1'))
fig.show()

[1 3 0] (74, 150, 3)


In [90]:
inds = (ydata[:,1]==3) & (ydata[:,0]==2) & (ydata[:,2]==0)
X = Xdata[inds]
y = ydata[inds]
Xp = Xpert[inds]
print(y[i], X.shape)
fig = go.Figure()
fig.add_trace(go.Scatter(y=X[i,:,0], mode='lines', name='x0'))
fig.add_trace(go.Scatter(y=X[i,:,1], mode='lines', name='y0'))
fig.add_trace(go.Scatter(y=X[i,:,2], mode='lines', name='z0'))
fig.add_trace(go.Scatter(y=Xp[i,:,0], mode='lines', name='x1'))
fig.add_trace(go.Scatter(y=Xp[i,:,1], mode='lines', name='y1'))
fig.add_trace(go.Scatter(y=Xp[i,:,2], mode='lines', name='z1'))
fig.show()

[2 3 0] (74, 150, 3)


In [92]:
J = 150
fs = 50
fmax = 25
f0s = np.arange(0.8, 0.9, 0.001)
X = X3
erro = []
razao = []
for f0 in f0s:
    L = np.floor(fmax/f0).astype(int)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    recon = theta.dot(coefs)
    erro.append(np.sqrt(np.mean((X - recon)**2)))
    razao.append(np.sum(recon**2)/np.sum(X**2))
fig = px.line(x=np.sort(f0s), y=razao,
        title=f'Erro de Reconstrução para Frequências Fundamentais Candidatas',
        labels={'x': 'Frequência Fundamental (Hz)', 'y': 'Erro de Reconstrução'}).show()
f0 = f0s[np.argmin(erro)]
print(f'Frequência fundamental estimada: {f0:.3f} Hz')

Frequência fundamental estimada: 0.812 Hz


In [79]:
Xpert = spectral_perturb(Xdata.transpose(0, 2, 1), gain_db=6., n_knots=20, preserve_dc=False).transpose(0, 2, 1)
i = 50
inds = (ydata[:,1]==3) & (ydata[:,0]==0) & (ydata[:,2]==0)
X1 = Xdata[inds][i]
Xp1 = Xpert[inds][i]
inds = (ydata[:,1]==3) & (ydata[:,0]==1) & (ydata[:,2]==0)
X2 = Xdata[inds][i]
Xp2 = Xpert[inds][i]
inds = (ydata[:,1]==3) & (ydata[:,0]==2) & (ydata[:,2]==0)
X3 = Xdata[inds][i]
Xp3 = Xpert[inds][i]
Xs = [X1, X2, X3]
Xp = [Xp1, Xp2, Xp3]
f0 = 0.848 # hand
f0 = 0.865 # chest
f0 = 0.861 # ankle

In [80]:
L = np.floor(fmax/f0).astype(int)
hAmps1 = []
for X in Xs:
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    recon = theta.dot(coefs)
    hAmps1.append(np.sqrt((coefs[:L,:]**2 + coefs[L:,:]**2).sum(axis=1)))
hAmps2 = []
for X in Xp:
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    recon = theta.dot(coefs)
    hAmps2.append(np.sqrt((coefs[:L,:]**2 + coefs[L:,:]**2).sum(axis=1)))
fig = go.Figure()
fig.add_trace(go.Scatter(y=hAmps1[0], mode='lines', name='Mão'))
fig.add_trace(go.Scatter(y=hAmps1[1], mode='lines', name='Peito'))
fig.add_trace(go.Scatter(y=hAmps1[2], mode='lines', name='Tornozelo'))
fig.add_trace(go.Scatter(y=hAmps2[0], mode='lines', name='MãoP'))
fig.add_trace(go.Scatter(y=hAmps2[1], mode='lines', name='PeitoP'))
fig.add_trace(go.Scatter(y=hAmps2[2], mode='lines', name='TornozeloP'))
fig.show()

In [82]:
px.line(20*np.log10(hAmps1[1]/hAmps1[0]))

In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/baseline_10_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
resultados = np.zeros((10,3,3))
for rep in tqdm(range(10)):
    for i, posi in enumerate(posis):
        enc = ChangEncoder().to(device)
        cla = ChangClassifier().to(device)
        nome = str(rep)+'_baseline_encoder_'+posi+'.pth'
        enc.load_state_dict(torch.load(path + nome, map_location=torch.device('cpu')))
        nome = str(rep)+'_baseline_classifier_'+posi+'.pth'
        cla.load_state_dict(torch.load(path + nome, map_location=torch.device('cpu')))
        for j in range(3):
            inds = ydata[:,0]==j
            X = Xdata[inds]
            y = ydata[inds][:,1]
            _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
            X_test  = torch.tensor(X_test, dtype=torch.float32)
            X_test = spectral_perturb(X_test.transpose(1,2), gain_db=24., n_knots=20, generator=None, preserve_dc=False)
            y_test  = torch.tensor(y_test, dtype=torch.long)
            test_loader = DataLoader(TensorDataset(X_test.transpose(1,2), y_test), batch_size=256)#, pin_memory=True)
            _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
            resultados[rep,i,j] = test_f1
print(), print()
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

100%|██████████| 10/10 [00:14<00:00,  1.42s/it]



hand:	44 +- 1 | 19 +- 2 | 12 +- 1 | 
chest:	14 +- 1 | 50 +- 2 | 9 +- 1 | 
ankle:	11 +- 2 | 9 +- 2 | 48 +- 2 | 


In [ ]:
# Resultados Baseline Reprodução
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	76 +- 2 | 27 +- 3 | 15 +- 2 | 
chest:	20 +- 2 | 82 +- 1 | 12 +- 2 | 
ankle:	13 +- 3 | 12 +- 3 | 80 +- 1 | 


## Resultados com filtros aleatórios preservando o nível DC

In [ ]:
# Resultados Filtros 6 dB preservando DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	74 +- 2 | 27 +- 3 | 15 +- 1 | 
chest:	19 +- 2 | 81 +- 1 | 11 +- 1 | 
ankle:	13 +- 3 | 12 +- 3 | 76 +- 1 | 


In [ ]:
# Resultados Filtros 12 dB preservando DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	68 +- 2 | 26 +- 3 | 14 +- 1 | 
chest:	18 +- 2 | 75 +- 1 | 11 +- 1 | 
ankle:	13 +- 3 | 11 +- 2 | 71 +- 2 | 


In [ ]:
# Resultados Filtros 18 dB preservando DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	58 +- 3 | 23 +- 3 | 14 +- 2 | 
chest:	16 +- 2 | 66 +- 1 | 11 +- 1 | 
ankle:	12 +- 2 | 11 +- 2 | 63 +- 2 | 


In [ ]:
# Resultados Filtros 24 dB preservando DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	47 +- 3 | 20 +- 2 | 13 +- 1 | 
chest:	13 +- 1 | 55 +- 1 | 9 +- 1 | 
ankle:	11 +- 2 | 11 +- 2 | 53 +- 2 | 


## Resultado com filtros aleatórios mudando o nível DC

In [ ]:
# Resultados Filtros 6 dB mudando o DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	73 +- 1 | 27 +- 3 | 15 +- 1 | 
chest:	19 +- 2 | 79 +- 1 | 11 +- 1 | 
ankle:	13 +- 3 | 12 +- 3 | 75 +- 2 | 


In [ ]:
# Resultados Filtros 12 dB mudando o DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	65 +- 2 | 25 +- 3 | 14 +- 2 | 
chest:	17 +- 2 | 71 +- 2 | 11 +- 1 | 
ankle:	12 +- 3 | 11 +- 3 | 68 +- 2 | 


In [ ]:
# Resultados Filtros 18 dB mudando o DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	54 +- 2 | 22 +- 2 | 13 +- 1 | 
chest:	15 +- 1 | 60 +- 1 | 10 +- 1 | 
ankle:	12 +- 2 | 10 +- 2 | 58 +- 2 | 


In [ ]:
# Resultados Filtros 24 dB mudando o DC
medias = (100*resultados.mean(axis=0)).round().astype(int)
desvio = (100*resultados.std(axis=0)).round().astype(int)
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{medias[i,j]} +- {desvio[i,j]}', end=' | ')
    print()

hand:	44 +- 1 | 19 +- 2 | 12 +- 1 | 
chest:	14 +- 1 | 50 +- 2 | 9 +- 1 | 
ankle:	11 +- 2 | 9 +- 2 | 48 +- 2 | 


# Treinamento com filtros aleatórios

In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/filtros_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])

X_filt = torch.tensor(Xdata, dtype=torch.float32)
X_filt = spectral_perturb(X_filt.transpose(1,2), gain_db=24., n_knots=20, generator=None, preserve_dc=True)
X_filt = X_filt.transpose(1,2).numpy()
rep = 0
for i, posi in enumerate(posis):
    inds = ydata[:,0]==i
    X = X_filt[inds]
    y = ydata[inds][:,1]
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
    X = Xdata[inds]
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test  = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
    enc, cla, history = train_model(train_loader, test_loader, device, n_epochs=100)
    nome = str(rep)+'_baseline_encoder_'+posi+'.pth'
    torch.save(enc.state_dict(), path + nome)
    nome = str(rep)+'_baseline_classifier_'+posi+'.pth'
    torch.save(cla.state_dict(), path + nome)

In [99]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/filtros_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
resultados = np.zeros((3,3))
rep = 0
for i, posi in enumerate(posis):
    enc = ChangEncoder().to(device)
    cla = ChangClassifier().to(device)
    nome = str(rep)+'_baseline_encoder_'+posi+'.pth'
    enc.load_state_dict(torch.load(path + nome))
    nome = str(rep)+'_baseline_classifier_'+posi+'.pth'
    cla.load_state_dict(torch.load(path + nome))
    for j in range(3):
        inds = ydata[:,0]==j
        X = Xdata[inds]
        y = ydata[inds][:,1]
        _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
        resultados[i,j] = test_f1
print(), print()
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()



hand:	0.66 | 0.21 | 0.25 | 
chest:	0.18 | 0.74 | 0.14 | 
ankle:	0.14 | 0.14 | 0.68 | 


In [16]:
X_train, X_TEST, y_train, y_TEST = train_test_split(Xdata, ydata, test_size=0.2, random_state=seeds[rep], stratify=ydata[:,1])
aux = []
yau = []
for gain in [6.,12.,18.]:
    X_filt = torch.tensor(X_train, dtype=torch.float32)
    X_filt = spectral_perturb(X_filt.transpose(1,2), gain_db=gain, n_knots=20, generator=None, preserve_dc=True)
    X_filt = X_filt.transpose(1,2).numpy()
    aux.append(X_filt)
    yau.append(y_train)
aux.append(X_train)
yau.append(y_train)
X_filts = np.vstack(aux)
y_filts = np.vstack(yau)

In [ ]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/filtros_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
rep = 0
for i, posi in enumerate(posis):
    inds = y_filts[:,0]==i
    X = X_filts[inds]
    y = y_filts[inds][:,1]
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=seeds[rep], stratify=y)
    inds = y_TEST[:,0]==i
    X_test = X_TEST[inds]
    y_test = y_TEST[inds][:,1]
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test  = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
    enc, cla, history = train_model(train_loader, test_loader, device, n_epochs=100)
    nome = str(rep)+'_filts_encoder_'+posi+'.pth'
    torch.save(enc.state_dict(), path + nome)
    nome = str(rep)+'_filts_classifier_'+posi+'.pth'
    torch.save(cla.state_dict(), path + nome)

In [18]:
path = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/filtros_PAMAP/'
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
resultados = np.zeros((3,3))
rep = 0
for i, posi in enumerate(posis):
    enc = ChangEncoder().to(device)
    cla = ChangClassifier().to(device)
    nome = str(rep)+'_filts_encoder_'+posi+'.pth'
    enc.load_state_dict(torch.load(path + nome))
    nome = str(rep)+'_filts_classifier_'+posi+'.pth'
    cla.load_state_dict(torch.load(path + nome))
    for j in range(3):
        inds = y_TEST[:,0]==j
        X_test = X_TEST[inds]
        y_test = y_TEST[inds][:,1]
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
        resultados[i,j] = test_f1
print(), print()
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()



hand:	0.78 | 0.27 | 0.18 | 
chest:	0.19 | 0.83 | 0.11 | 
ankle:	0.17 | 0.14 | 0.79 | 


## Alguns resultados

In [97]:
# Resultados Filtros 6 dB preservando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.74 | 0.3 | 0.18 | 
chest:	0.19 | 0.82 | 0.09 | 
ankle:	0.11 | 0.14 | 0.8 | 


In [ ]:
# Resultados Filtros 12 dB preservando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.72 | 0.29 | 0.16 | 
chest:	0.22 | 0.8 | 0.08 | 
ankle:	0.15 | 0.17 | 0.75 | 


In [ ]:
# Resultados Filtros 18 dB preservando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.69 | 0.23 | 0.2 | 
chest:	0.16 | 0.74 | 0.11 | 
ankle:	0.2 | 0.12 | 0.7 | 


In [100]:
# Resultados Filtros 24 dB preservando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.66 | 0.21 | 0.25 | 
chest:	0.18 | 0.74 | 0.14 | 
ankle:	0.14 | 0.14 | 0.68 | 


In [ ]:
# Resultados Filtros 6 dB mudando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.75 | 0.3 | 0.14 | 
chest:	0.16 | 0.81 | 0.1 | 
ankle:	0.11 | 0.11 | 0.78 | 


In [ ]:
# Resultados Filtros 12 dB mudando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.7 | 0.28 | 0.18 | 
chest:	0.16 | 0.76 | 0.13 | 
ankle:	0.12 | 0.1 | 0.74 | 


In [ ]:
# Resultados Filtros 18 dB mudando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.63 | 0.25 | 0.14 | 
chest:	0.16 | 0.68 | 0.13 | 
ankle:	0.17 | 0.14 | 0.68 | 


In [ ]:
# Resultados Filtros 24 dB mudando o DC
for i in range(3):
    print(posis[i], end=':\t')
    for j in range(3):
        print(f'{resultados[i,j].round(2)}', end=' | ')
    print()

hand:	0.57 +- 1 | 0.25 +- 2 | 0.19 +- 1 | 
chest:	0.16 +- 1 | 0.61 +- 2 | 0.12 +- 1 | 
ankle:	0.19 +- 2 | 0.13 +- 2 | 0.6 +- 2 | 


# Adaptando com MMD

In [19]:
def mmd2u(x, y, c):
    n = x.shape[0]
    m = y.shape[0]
    xy = torch.vstack((x,y))
    dists = torch.cdist(xy, xy)
    k = torch.exp( (-1/(2*c)) * dists**2 )
    k_x = torch.triu(k[:n, :n], diagonal=1)
    k_y = torch.triu(k[n:, n:], diagonal=1)
    k_xy = k[:n, n:]
    mmd = 2*k_x.sum()/(n*(n-1)) + 2*k_y.sum()/(m*(m-1)) - 2*k_xy.sum()/(n*m)
    return mmd

In [20]:
def train_feature_matching(Xs_train, ys_train, Xt_train, Xs_val, ys_val, Xt_val, encoder, classifier):
    if encoder is None:
        encoder = ChangEncoder().to(device)
    if classifier is None:
        classifier = ChangClassifier().to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt_cls = torch.optim.Adam(list(encoder.parameters()) + list(classifier.parameters()), lr=1e-3)
    opt_mmd = torch.optim.Adam(encoder.parameters(), lr=1e-3)
    history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": [],
               "val_mmd": []}
    Ns = len(Xs_train)
    Nt = len(Xt_train)
    for epoch in range(epochs):
        encoder.train()
        classifier.train()
        perm_s = torch.randperm(Ns, device=device)
        perm_t = torch.randperm(Nt, device=device)
        Xs = Xs_train[perm_s]
        ys = ys_train[perm_s]
        Xt = Xt_train[perm_t]
        n_batches = min(Ns, Nt) // batch_size
        pbar = tqdm(range(n_batches), desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for b in pbar:
            i0 = b * batch_size
            i1 = i0 + batch_size
            Xsb = Xs[i0:i1]
            ysb = ys[i0:i1]
            Xtb = Xt[i0:i1]
            # PASSO 1: classificação
            opt_cls.zero_grad()
            feat = encoder(Xsb)
            logits = classifier(feat)
            loss_cls = loss_fn(logits, ysb)
            loss_cls.backward()
            opt_cls.step()
            # PASSO 2: feature matching
            opt_mmd.zero_grad()
            feat_s = encoder(Xsb)
            feat_t = encoder(Xtb)
            loss_mmd = 0.0
            for sigma2 in vals:
                loss_mmd += mmd2u(feat_s, feat_t, sigma2)
            loss_mmd.backward()
            opt_mmd.step()
            pbar.set_postfix(cls=f"{loss_cls.item():.4f}", mmd=f"{loss_mmd.item():.4f}")
        encoder.eval()
        classifier.eval()
        with torch.no_grad():
            feat_s = encoder(Xs_train)
            logits = classifier(feat_s)
            train_loss = loss_fn(logits, ys_train)
            train_pred = logits.argmax(1)
            train_f1 = multiclass_f1_score(train_pred, ys_train, num_classes=8)
            feat_s = encoder(Xs_val)
            feat_t = encoder(Xt_val)
            logits = classifier(feat_s)
            val_loss = loss_fn(logits, ys_val)
            val_pred = logits.argmax(1)
            val_f1 = multiclass_f1_score(val_pred, ys_val, num_classes=8)
            val_mmd = 0.0
            for sigma2 in vals:
                val_mmd += mmd2u(feat_s, feat_t, sigma2)
        history["train_loss"].append(train_loss.item())
        history["train_f1"].append(train_f1.item())
        history["val_loss"].append(val_loss.item())
        history["val_f1"].append(val_f1.item())
        history["val_mmd"].append(val_mmd.item())
        # print(
        #     f"Epoch {epoch+1:2d} | "
        #     f"train F1={train_f1:.4f} | "
        #     f"val F1={val_f1:.4f}"
        # )
    return encoder, classifier, history

In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/adapt_PAMAP/'
vals = [0.0001, 0.001, 0.01, 0.1, 1, 5, 10, 15, 20, 25, 30, 100]
batch_size = 125
epochs = 100
for i in range(3):
    for j in range(3):
        if i==j:
            continue
        print('Adaptando modelo '+posis[i]+' para '+posis[j])
        enc2 = ChangEncoder().to(device)
        enc2.load_state_dict(torch.load(pasta+'baseline_encoder_'+posis[i]+'.pth'))
        cla2 = ChangClassifier().to(device)
        cla2.load_state_dict(torch.load(pasta+'baseline_classifier_'+posis[i]+'.pth'))
        inds = ydata[:,0]==i
        Xs = Xdata[inds]
        ys = ydata[inds][:,1]
        Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs, ys, test_size=0.2, random_state=1, stratify=ys)
        Xs_train = torch.tensor(Xs_train, dtype=torch.float32).to(device)
        Xs_test  = torch.tensor(Xs_test, dtype=torch.float32).to(device)
        ys_train = torch.tensor(ys_train, dtype=torch.long).to(device)
        ys_test  = torch.tensor(ys_test, dtype=torch.long).to(device)
        inds = ydata[:,0]==j
        Xt = Xdata[inds]
        yt = ydata[inds][:,1]
        Xt_train, Xt_test, yt_train, yt_test = train_test_split(Xt, yt, test_size=0.2, random_state=1, stratify=yt)
        Xt_train = torch.tensor(Xt_train, dtype=torch.float32).to(device)
        Xt_test  = torch.tensor(Xt_test, dtype=torch.float32).to(device)
        yt_train = torch.tensor(yt_train, dtype=torch.long).to(device)
        yt_test  = torch.tensor(yt_test, dtype=torch.long).to(device)
        enc2, cla2, history = train_feature_matching(Xs_train, ys_train, Xt_train, Xs_test, ys_test, Xt_test, enc2, cla2)
        nome = posis[i]+'_adapt_encoder_'+posis[j]+'.pth'
        torch.save(enc2.state_dict(), nome)
        nome = posis[i]+'_adapt_classifier_'+posis[j]+'.pth'
        torch.save(cla2.state_dict(), nome)
        nome = posis[i]+'_adapt_history_'+posis[j]+'.csv'
        pd.DataFrame(history).to_csv(nome)

In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/adapt_PAMAP/'
vals = [0.0001, 0.001, 0.01, 0.1, 1, 5, 10, 15, 20, 25, 30, 100]
adapt = np.zeros((3,3))
for i in range(3):
    for j in range(3):
        if i==j:
            continue
        enc2 = ChangEncoder().to(device)
        nome = posis[i]+'_adapt_encoder_'+posis[j]+'.pth'
        enc2.load_state_dict(torch.load(pasta+nome))
        cla2 = ChangClassifier().to(device)
        nome = posis[i]+'_adapt_classifier_'+posis[j]+'.pth'
        cla2.load_state_dict(torch.load(pasta+nome))

        inds = ydata[:,0]==j
        Xs = Xdata[inds]
        ys = ydata[inds][:,1]
        _, Xs_test, _, ys_test = train_test_split(Xs, ys, test_size=0.2, random_state=1, stratify=ys)
        Xs_test  = torch.tensor(Xs_test, dtype=torch.float32).to(device)
        ys_test  = torch.tensor(ys_test, dtype=torch.long).to(device)
        test_loader = DataLoader(TensorDataset(Xs_test, ys_test), batch_size=256, shuffle=False)
        _, test_f1 = evaluate(enc2, cla2, test_loader, nn.CrossEntropyLoss(), device)
        adapt[i,j] = test_f1

In [ ]:
df = pd.DataFrame((adapt*100).round(1), columns=posis, index=posis)
df

,hand,chest,ankle
hand,0.0,40.6,26.6
chest,34.5,0.0,31.3
ankle,19.0,45.0,0.0


In [ ]:
df2 = pd.DataFrame([[0., 27.9, 13.7], [32., 0., 26.6], [15.6, 14.5, 0.]], columns=posis, index=posis)
df2

NameError: name 'pd' is not defined